In [ ]:
"""
Optimized MSI Processing Pipeline with Memory Management and Exact DPI Calculation
"""

import os
import gc
import psutil
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend to prevent memory leaks
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial

# =============================================================================
# CONFIGURATION SECTION
# =============================================================================

class Config:
    """Configuration settings for memory and CPU usage"""
    # Resource limits
    MAX_MEMORY_GB = 160  # Maximum memory usage in GB (adjust based on your system)
    NUM_CORES = 78  # Number of CPU cores to use (None = all available - 1)
    BATCH_SIZE = 156  # Number of plots to process before garbage collection
    
    # Input/Output paths
    MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain/"
    COMMON_MZS_CSV = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/common_mz_clusters_improved.csv"
    OUTPUT_DIR = "/home/ajarrah/PhD_Thesis/chapter_4/images/lipids_60/"
    
    # Processing parameters
    SQUARE_SPACING = 60  # µm center-to-center
    MZ_TOLERANCE = 0.01
    VMAX_PERCENTILE = 99.9
    SQUARE_SIZE = 60  # µm per square
    PIXEL_SIZE_UM = 60  # Desired pixel size in micrometers for plotting
    
    # Sample IDs
    SAMPLE_IDS = [
        "yc_1", "yc_2", "yc_3", "yc_4",
        "yad_1", "yad_2", "yad_3", "yad_4",
        "ac_1", "ac_2", "ac_3", "ac_4",
        "aad_1", "aad_2", "aad_3", "aad_4"
        
    ]
    
    MSI_ORDER = [
        'yc_1', 'yc_2', 'yc_3', 'yc_4',
        'yad_1', 'yad_2', 'yad_3', 'yad_4',
        'ac_1', 'ac_2', 'ac_3', 'ac_4',
        'aad_1', 'aad_2', 'aad_3', 'aad_4'
    ]

# =============================================================================
# CUSTOM COLORMAP
# =============================================================================

CUSTOM_CMAP = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),      # dark gray
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),     # navy
    (0.15, "#0000FF"),     # blue
    (0.30, "#8000FF"),     # purple-ish
    (0.45, "#FF0000"),     # red
    (0.60, "#FF8000"),     # orange
    (0.75, "#FFFF00"),     # yellow
    (1.0, "#FFFFFF")       # white
])

# =============================================================================
# MEMORY MONITORING UTILITIES
# =============================================================================

def check_memory_usage():
    """Check current memory usage"""
    process = psutil.Process(os.getpid())
    mem_gb = process.memory_info().rss / (1024 ** 3)
    return mem_gb

def force_garbage_collection():
    """Force garbage collection and clear matplotlib cache"""
    plt.close('all')
    gc.collect()

# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================

def load_sample(sample_id, input_folder):
    """Load a single h5ad sample with error handling"""
    filepath = os.path.join(input_folder, f"halfbrain_{sample_id}_filtered.h5ad")
    try:
        adata = sc.read_h5ad(filepath)
        print(f"✓ Loaded {sample_id}")
        return adata
    except Exception as e:
        print(f"✗ Failed to load {sample_id}: {e}")
        return None

def preprocess_sample(adata, pixel_spacing):
    """Add TIC and spatial coordinates to AnnData object"""
    # Calculate TIC
    tic = adata.X.sum(axis=1)
    if hasattr(tic, "A1"):  # sparse matrix
        tic = tic.A1
    else:
        tic = np.asarray(tic).flatten()
    adata.obs["Processed_TIC"] = tic
    
    # Add spatial coordinates in µm
    spatial = adata.obsm['spatial']
    array_row = spatial[:, 1]
    array_col = spatial[:, 0]
    adata.obs['x_um'] = array_col * pixel_spacing
    adata.obs['y_um'] = array_row * pixel_spacing
    
    return adata

def load_and_preprocess_all_samples(config):
    """Load all samples and preprocess them"""
    print("=" * 80)
    print("LOADING AND PREPROCESSING SAMPLES")
    print("=" * 80)
    
    sample_map = {}
    for sample_id in config.SAMPLE_IDS:
        adata = load_sample(sample_id, config.MSI_INPUT_FOLDER)
        if adata is not None:
            adata = preprocess_sample(adata, config.SQUARE_SPACING)
            sample_map[sample_id] = adata
            
        # Monitor memory
        mem_gb = check_memory_usage()
        if mem_gb > config.MAX_MEMORY_GB * 0.8:
            print(f"⚠️ Memory usage high ({mem_gb:.2f} GB), forcing cleanup...")
            force_garbage_collection()
    
    print(f"\n✓ Loaded {len(sample_map)}/{len(config.SAMPLE_IDS)} samples successfully\n")
    return sample_map

def load_common_mz_data(csv_path, msi_order):
    """Load and pivot common m/z data"""
    print("Loading common m/z data...")
    common_mz_df = pd.read_csv(csv_path)
    
    # Create sample_id column
    common_mz_df['sample_id'] = (
        common_mz_df['group'].str.lower() + '_' + 
        common_mz_df['sample'].astype(str)
    )
    
    # Pivot and sort
    pivot_df = common_mz_df.pivot(
        index='sample_id', 
        columns='common_group_name', 
        values='mzs'
    )
    pivot_df_sorted = pivot_df.loc[pivot_df.index.intersection(msi_order)]
    pivot_df_sorted = pivot_df_sorted.loc[msi_order]
    
    print(f"✓ Loaded m/z data for {len(pivot_df_sorted)} samples\n")
    return pivot_df_sorted

# =============================================================================
# PLOTTING FUNCTION WITH EXACT DPI CALCULATION
# =============================================================================

def msi_plot_producer(
    adata,
    mz_val,
    sample_name,
    output_dir,
    tol=0.01,
    vmax_percentile=99.8,
    square_size=60,
    pixel_size_um=5,
    cmap=CUSTOM_CMAP,
    show_colorbar=False,
    show_axis=False,
    show_title=False,
    background_color="#00BF00"
):
    """
    Plot TIC-normalized intensity for one m/z and one sample.
    Image size and DPI are calculated exactly based on data dimensions and pixel size.
    
    Args:
        square_size: Physical size of each pixel in micrometers (µm)
    """
    try:
        # Extract m/z index
        mz_axis = adata.var_names.astype(float).values
        mz_diff = np.abs(mz_axis - mz_val)
        if np.min(mz_diff) > tol:
            return False, f"m/z {mz_val} not found within tolerance {tol}"
        mz_index = np.argmin(mz_diff)
        
        # Extract intensities
        intensities = (
            adata.X[:, mz_index].toarray().flatten()
            if hasattr(adata.X, "toarray")
            else adata.X[:, mz_index]
        )
        
        # Extract coordinates & TIC
        x = adata.obs["x_um"].values
        y = adata.obs["y_um"].values
        tic = adata.obs["Processed_TIC"].values
        
        # Normalize by TIC
        normalized_intensities = (intensities / tic) * 100
        normalized_intensities = np.nan_to_num(
            normalized_intensities, nan=0.0, posinf=0.0, neginf=0.0
        )
        
        # Build grid
        x_unique = np.unique(x)
        y_unique = np.unique(y)
        x_size = len(x_unique)  # Number of pixels in x
        y_size = len(y_unique)  # Number of pixels in y
        grid = np.full((y_size, x_size), np.nan)
        
        for xi, yi, val in zip(x, y, normalized_intensities):
            x_idx = np.where(x_unique == xi)[0][0]
            y_idx = np.where(y_unique == yi)[0][0]
            grid[y_idx, x_idx] = val
        
        # Color scaling
        vmin_value = np.nanmin(normalized_intensities)
        vmax_value = np.percentile(normalized_intensities, vmax_percentile)
        
        # =====================================================================
        # EXACT DPI AND SIZE CALCULATION
        # =====================================================================
        # Physical dimensions in micrometers
        physical_width_um = x_size * square_size
        physical_height_um = y_size * square_size
        
        # Convert to inches (1 inch = 25,400 µm)
        UM_PER_INCH = 25400.0
        physical_width_inch = physical_width_um / UM_PER_INCH
        physical_height_inch = physical_height_um / UM_PER_INCH
        
        # Calculate DPI so that:
        # image_pixels = data_pixels (1:1 mapping)
        # DPI = pixels / inches
        dpi = UM_PER_INCH / pixel_size_um  # pixels per inch in x 
        
        # Figsize in inches (matplotlib requires inches)
        figsize = (physical_width_inch, physical_height_inch)
        
        # =====================================================================
        # CREATE FIGURE WITH EXACT DIMENSIONS
        # =====================================================================
        
        extent = [x.min(), x.max() + 2*square_size, y.min(), y.max() + 2*square_size]
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        fig.patch.set_facecolor(background_color)
        
        im = ax.imshow(
            grid,
            cmap=cmap,
            origin="lower",
            extent=extent,
            vmin=vmin_value,
            vmax=vmax_value,
            interpolation="none",
        )
        ax.invert_xaxis()
        ax.set_aspect("equal")
        
        if show_axis:
            ax.set_xlabel("x (µm)")
            ax.set_ylabel("y (µm)")
        else:
            ax.axis("off")
        
        if show_title:
            ax.set_title(f"m/z {mz_val:.4f} normalized intensity")
        
        if show_colorbar:
            cbar = fig.colorbar(im, ax=ax)
            cbar.set_label("Normalized intensity (TIC%)")
        
        # Save image with exact DPI
        sample_dir = os.path.join(output_dir, sample_name)
        os.makedirs(sample_dir, exist_ok=True)
        filename = f"{mz_val:.4f}|{sample_name}|{vmin_value:.2f}|{vmax_value:.2f}.png"
        filepath = os.path.join(sample_dir, filename)
        
        # Save with calculated DPI and no padding
        fig.savefig(
            filepath, 
            dpi=dpi,  # Use calculated DPI
            bbox_inches="tight",
            pad_inches=0,  # No padding
            transparent=False,
            facecolor=background_color
        )
        
        # Clean up immediately
        plt.close(fig)
        del fig, ax, im, grid, normalized_intensities
        
        return True, f"✅ {sample_name} | m/z {mz_val:.4f} | {x_size}×{y_size}px | DPI:{dpi:.1f}"
        
    except Exception as e:
        plt.close('all')
        return False, f"⚠️ Failed {sample_name} | m/z {mz_val:.4f}: {e}"

# =============================================================================
# PARALLEL PROCESSING WITH SERIALIZATION
# =============================================================================

def process_mz_wrapper(args):
    """
    Wrapper function for multiprocessing.
    Each process loads its own copy of the data to avoid serialization issues.
    """
    sample_id, mz_val, config = args
    
    try:
        # Load sample in this process
        adata = load_sample(sample_id, config.MSI_INPUT_FOLDER)
        if adata is None:
            return False, f"⚠️ Failed to load {sample_id}"
        
        adata = preprocess_sample(adata, config.SQUARE_SPACING)
        
        # Generate plot
        success, msg = msi_plot_producer(
            adata=adata,
            mz_val=mz_val,
            sample_name=sample_id,
            output_dir=config.OUTPUT_DIR,
            tol=config.MZ_TOLERANCE,
            vmax_percentile=config.VMAX_PERCENTILE,
            square_size=config.SQUARE_SIZE,
            pixel_size_um=config.PIXEL_SIZE_UM,
            cmap=CUSTOM_CMAP,
            show_colorbar=False,
            show_axis=False,
            show_title=False,
            background_color="#00BF00"
        )
        
        # Clean up
        del adata
        force_garbage_collection()
        
        return success, msg
        
    except Exception as e:
        return False, f"⚠️ Error {sample_id} | m/z {mz_val:.4f}: {e}"

# =============================================================================
# MAIN PROCESSING PIPELINE
# =============================================================================

def main():
    """Main processing pipeline"""
    # Initialize configuration
    config = Config()
    
    # Set number of cores
    if config.NUM_CORES is None:
        max_workers = max(1, os.cpu_count() - 1)
    else:
        max_workers = min(config.NUM_CORES, os.cpu_count())
    
    print("=" * 80)
    print("MSI PROCESSING PIPELINE - EXACT DPI CALCULATION")
    print("=" * 80)
    print(f"Max Memory: {config.MAX_MEMORY_GB} GB")
    print(f"CPU Cores: {max_workers}")
    print(f"Batch Size: {config.BATCH_SIZE}")
    print(f"Pixel Size: {config.SQUARE_SIZE} µm")
    print("=" * 80 + "\n")
    
    # Load common m/z data
    pivot_df = load_common_mz_data(config.COMMON_MZS_CSV, config.MSI_ORDER)
    
    # Prepare all tasks
    tasks = []
    for sample_id in config.SAMPLE_IDS:
        if sample_id not in pivot_df.index:
            print(f"⚠️ Sample '{sample_id}' not in pivot table — skipping.")
            continue
        
        mz_values = pivot_df.loc[sample_id].values.astype(float)
        mz_values = mz_values[~np.isnan(mz_values)]
        
        for mz_val in mz_values:
            tasks.append((sample_id, mz_val, config))
    
    print(f"Total tasks to process: {len(tasks)}\n")
    print("=" * 80)
    print("PROCESSING")
    print("=" * 80 + "\n")
    
    # Process in batches to manage memory
    success_count = 0
    fail_count = 0
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = [executor.submit(process_mz_wrapper, task) for task in tasks]
        
        # Process results
        for i, future in enumerate(as_completed(futures), 1):
            success, msg = future.result()
            if success:
                success_count += 1
            else:
                fail_count += 1
            
            print(f"[{i}/{len(tasks)}] {msg}")
            
            # Periodic garbage collection
            if i % config.BATCH_SIZE == 0:
                force_garbage_collection()
                mem_gb = check_memory_usage()
                print(f"  └─ Memory usage: {mem_gb:.2f} GB")
    
    # Final cleanup
    force_garbage_collection()
    
    print("\n" + "=" * 80)
    print("PROCESSING COMPLETE")
    print("=" * 80)
    print(f"✅ Success: {success_count}")
    print(f"⚠️ Failed: {fail_count}")
    print(f"📊 Total: {len(tasks)}")
    print("=" * 80)

if __name__ == "__main__":
    main()

MSI PROCESSING PIPELINE - EXACT DPI CALCULATION
Max Memory: 160 GB
CPU Cores: 78
Batch Size: 156
Pixel Size: 60 µm

Loading common m/z data...
✓ Loaded m/z data for 16 samples

Total tasks to process: 8448

PROCESSING

✓ Loaded yc_1
✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1
✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_

In [1]:
"""
Optimized MSI Processing Pipeline with Memory Management and Exact DPI Calculation
Modified to use common m/z names instead of m/z values in filenames
"""

import os
import gc
import psutil
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend to prevent memory leaks
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial

# =============================================================================
# CONFIGURATION SECTION
# =============================================================================

class Config:
    """Configuration settings for memory and CPU usage"""
    # Resource limits
    MAX_MEMORY_GB = 160  # Maximum memory usage in GB (adjust based on your system)
    NUM_CORES = 78  # Number of CPU cores to use (None = all available - 1)
    BATCH_SIZE = 156  # Number of plots to process before garbage collection
    
    # Input/Output paths
    MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain/"
    COMMON_MZS_CSV = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/common_mz_clusters_improved.csv"
    OUTPUT_DIR = "/home/ajarrah/PhD_Thesis/chapter_4/images/lipids_60_common/"
    
    # Processing parameters
    SQUARE_SPACING = 60  # µm center-to-center
    MZ_TOLERANCE = 0.01
    VMAX_PERCENTILE = 99.9
    SQUARE_SIZE = 60  # µm per square
    PIXEL_SIZE_UM = 60  # Desired pixel size in micrometers for plotting
    
    # Sample IDs
    SAMPLE_IDS = [
        "yc_1", "yc_2", "yc_3", "yc_4",
        "yad_1", "yad_2", "yad_3", "yad_4",
        "ac_1", "ac_2", "ac_3", "ac_4",
        "aad_1", "aad_2", "aad_3", "aad_4"
        
    ]
    
    MSI_ORDER = [
        'yc_1', 'yc_2', 'yc_3', 'yc_4',
        'yad_1', 'yad_2', 'yad_3', 'yad_4',
        'ac_1', 'ac_2', 'ac_3', 'ac_4',
        'aad_1', 'aad_2', 'aad_3', 'aad_4'
    ]

# =============================================================================
# CUSTOM COLORMAP
# =============================================================================

CUSTOM_CMAP = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),      # dark gray
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),     # navy
    (0.15, "#0000FF"),     # blue
    (0.30, "#8000FF"),     # purple-ish
    (0.45, "#FF0000"),     # red
    (0.60, "#FF8000"),     # orange
    (0.75, "#FFFF00"),     # yellow
    (1.0, "#FFFFFF")       # white
])

# =============================================================================
# MEMORY MONITORING UTILITIES
# =============================================================================

def check_memory_usage():
    """Check current memory usage"""
    process = psutil.Process(os.getpid())
    mem_gb = process.memory_info().rss / (1024 ** 3)
    return mem_gb

def force_garbage_collection():
    """Force garbage collection and clear matplotlib cache"""
    plt.close('all')
    gc.collect()

# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================

def load_sample(sample_id, input_folder):
    """Load a single h5ad sample with error handling"""
    filepath = os.path.join(input_folder, f"halfbrain_{sample_id}_filtered.h5ad")
    try:
        adata = sc.read_h5ad(filepath)
        print(f"✓ Loaded {sample_id}")
        return adata
    except Exception as e:
        print(f"✗ Failed to load {sample_id}: {e}")
        return None

def preprocess_sample(adata, pixel_spacing):
    """Add TIC and spatial coordinates to AnnData object"""
    # Calculate TIC
    tic = adata.X.sum(axis=1)
    if hasattr(tic, "A1"):  # sparse matrix
        tic = tic.A1
    else:
        tic = np.asarray(tic).flatten()
    adata.obs["Processed_TIC"] = tic
    
    # Add spatial coordinates in µm
    spatial = adata.obsm['spatial']
    array_row = spatial[:, 1]
    array_col = spatial[:, 0]
    adata.obs['x_um'] = array_col * pixel_spacing
    adata.obs['y_um'] = array_row * pixel_spacing
    
    return adata

def load_and_preprocess_all_samples(config):
    """Load all samples and preprocess them"""
    print("=" * 80)
    print("LOADING AND PREPROCESSING SAMPLES")
    print("=" * 80)
    
    sample_map = {}
    for sample_id in config.SAMPLE_IDS:
        adata = load_sample(sample_id, config.MSI_INPUT_FOLDER)
        if adata is not None:
            adata = preprocess_sample(adata, config.SQUARE_SPACING)
            sample_map[sample_id] = adata
            
        # Monitor memory
        mem_gb = check_memory_usage()
        if mem_gb > config.MAX_MEMORY_GB * 0.8:
            print(f"⚠️ Memory usage high ({mem_gb:.2f} GB), forcing cleanup...")
            force_garbage_collection()
    
    print(f"\n✓ Loaded {len(sample_map)}/{len(config.SAMPLE_IDS)} samples successfully\n")
    return sample_map

def load_common_mz_data(csv_path, msi_order):
    """Load common m/z data and return both pivot table and name mapping"""
    print("Loading common m/z data...")
    common_mz_df = pd.read_csv(csv_path)
    
    # Create sample_id column
    common_mz_df['sample_id'] = (
        common_mz_df['group'].str.lower() + '_' + 
        common_mz_df['sample'].astype(str)
    )
    
    # Create mapping of (sample_id, mz_value) -> common_group_name
    mz_name_map = {}
    for _, row in common_mz_df.iterrows():
        key = (row['sample_id'], row['mzs'])
        mz_name_map[key] = row['common_group_name']
    
    # Pivot and sort
    pivot_df = common_mz_df.pivot(
        index='sample_id', 
        columns='common_group_name', 
        values='mzs'
    )
    pivot_df_sorted = pivot_df.loc[pivot_df.index.intersection(msi_order)]
    pivot_df_sorted = pivot_df_sorted.loc[msi_order]
    
    print(f"✓ Loaded m/z data for {len(pivot_df_sorted)} samples\n")
    return pivot_df_sorted, mz_name_map

# =============================================================================
# PLOTTING FUNCTION WITH EXACT DPI CALCULATION
# =============================================================================

def msi_plot_producer(
    adata,
    mz_val,
    common_group_name,
    sample_name,
    output_dir,
    tol=0.01,
    vmax_percentile=99.8,
    square_size=60,
    pixel_size_um=5,
    cmap=CUSTOM_CMAP,
    show_colorbar=False,
    show_axis=False,
    show_title=False,
    background_color="#00BF00"
):
    """
    Plot TIC-normalized intensity for one m/z and one sample.
    Image size and DPI are calculated exactly based on data dimensions and pixel size.
    
    Args:
        common_group_name: Common name for the m/z cluster (used in filename)
        square_size: Physical size of each pixel in micrometers (µm)
    """
    try:
        # Extract m/z index
        mz_axis = adata.var_names.astype(float).values
        mz_diff = np.abs(mz_axis - mz_val)
        if np.min(mz_diff) > tol:
            return False, f"m/z {mz_val} not found within tolerance {tol}"
        mz_index = np.argmin(mz_diff)
        
        # Extract intensities
        intensities = (
            adata.X[:, mz_index].toarray().flatten()
            if hasattr(adata.X, "toarray")
            else adata.X[:, mz_index]
        )
        
        # Extract coordinates & TIC
        x = adata.obs["x_um"].values
        y = adata.obs["y_um"].values
        tic = adata.obs["Processed_TIC"].values
        
        # Normalize by TIC
        normalized_intensities = (intensities / tic) * 100
        normalized_intensities = np.nan_to_num(
            normalized_intensities, nan=0.0, posinf=0.0, neginf=0.0
        )
        
        # Build grid
        x_unique = np.unique(x)
        y_unique = np.unique(y)
        x_size = len(x_unique)  # Number of pixels in x
        y_size = len(y_unique)  # Number of pixels in y
        grid = np.full((y_size, x_size), np.nan)
        
        for xi, yi, val in zip(x, y, normalized_intensities):
            x_idx = np.where(x_unique == xi)[0][0]
            y_idx = np.where(y_unique == yi)[0][0]
            grid[y_idx, x_idx] = val
        
        # Color scaling
        vmin_value = np.nanmin(normalized_intensities)
        vmax_value = np.percentile(normalized_intensities, vmax_percentile)
        
        # =====================================================================
        # EXACT DPI AND SIZE CALCULATION
        # =====================================================================
        # Physical dimensions in micrometers
        physical_width_um = x_size * square_size
        physical_height_um = y_size * square_size
        
        # Convert to inches (1 inch = 25,400 µm)
        UM_PER_INCH = 25400.0
        physical_width_inch = physical_width_um / UM_PER_INCH
        physical_height_inch = physical_height_um / UM_PER_INCH
        
        # Calculate DPI so that:
        # image_pixels = data_pixels (1:1 mapping)
        # DPI = pixels / inches
        dpi = UM_PER_INCH / pixel_size_um  # pixels per inch in x 
        
        # Figsize in inches (matplotlib requires inches)
        figsize = (physical_width_inch, physical_height_inch)
        
        # =====================================================================
        # CREATE FIGURE WITH EXACT DIMENSIONS
        # =====================================================================
        
        extent = [x.min(), x.max() + 2*square_size, y.min(), y.max() + 2*square_size]
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        fig.patch.set_facecolor(background_color)
        
        im = ax.imshow(
            grid,
            cmap=cmap,
            origin="lower",
            extent=extent,
            vmin=vmin_value,
            vmax=vmax_value,
            interpolation="none",
        )
        ax.invert_xaxis()
        ax.set_aspect("equal")
        
        if show_axis:
            ax.set_xlabel("x (µm)")
            ax.set_ylabel("y (µm)")
        else:
            ax.axis("off")
        
        if show_title:
            ax.set_title(f"{common_group_name} (m/z {mz_val:.4f}) normalized intensity")
        
        if show_colorbar:
            cbar = fig.colorbar(im, ax=ax)
            cbar.set_label("Normalized intensity (TIC%)")
        
        # Save image with common_group_name in filename
        sample_dir = os.path.join(output_dir, sample_name)
        os.makedirs(sample_dir, exist_ok=True)
        filename = f"{common_group_name}|{sample_name}|{vmin_value:.2f}|{vmax_value:.2f}.png"
        filepath = os.path.join(sample_dir, filename)
        
        # Save with calculated DPI and no padding
        fig.savefig(
            filepath, 
            dpi=dpi,  # Use calculated DPI
            bbox_inches="tight",
            pad_inches=0,  # No padding
            transparent=False,
            facecolor=background_color
        )
        
        # Clean up immediately
        plt.close(fig)
        del fig, ax, im, grid, normalized_intensities
        
        return True, f"✅ {sample_name} | {common_group_name} (m/z {mz_val:.4f}) | {x_size}×{y_size}px | DPI:{dpi:.1f}"
        
    except Exception as e:
        plt.close('all')
        return False, f"⚠️ Failed {sample_name} | {common_group_name} (m/z {mz_val:.4f}): {e}"

# =============================================================================
# PARALLEL PROCESSING WITH SERIALIZATION
# =============================================================================

def process_mz_wrapper(args):
    """
    Wrapper function for multiprocessing.
    Each process loads its own copy of the data to avoid serialization issues.
    """
    sample_id, mz_val, common_group_name, config = args
    
    try:
        # Load sample in this process
        adata = load_sample(sample_id, config.MSI_INPUT_FOLDER)
        if adata is None:
            return False, f"⚠️ Failed to load {sample_id}"
        
        adata = preprocess_sample(adata, config.SQUARE_SPACING)
        
        # Generate plot
        success, msg = msi_plot_producer(
            adata=adata,
            mz_val=mz_val,
            common_group_name=common_group_name,
            sample_name=sample_id,
            output_dir=config.OUTPUT_DIR,
            tol=config.MZ_TOLERANCE,
            vmax_percentile=config.VMAX_PERCENTILE,
            square_size=config.SQUARE_SIZE,
            pixel_size_um=config.PIXEL_SIZE_UM,
            cmap=CUSTOM_CMAP,
            show_colorbar=False,
            show_axis=False,
            show_title=False,
            background_color="#00BF00"
        )
        
        # Clean up
        del adata
        force_garbage_collection()
        
        return success, msg
        
    except Exception as e:
        return False, f"⚠️ Error {sample_id} | {common_group_name} (m/z {mz_val:.4f}): {e}"

# =============================================================================
# MAIN PROCESSING PIPELINE
# =============================================================================

def main():
    """Main processing pipeline"""
    # Initialize configuration
    config = Config()
    
    # Set number of cores
    if config.NUM_CORES is None:
        max_workers = max(1, os.cpu_count() - 1)
    else:
        max_workers = min(config.NUM_CORES, os.cpu_count())
    
    print("=" * 80)
    print("MSI PROCESSING PIPELINE - EXACT DPI CALCULATION")
    print("=" * 80)
    print(f"Max Memory: {config.MAX_MEMORY_GB} GB")
    print(f"CPU Cores: {max_workers}")
    print(f"Batch Size: {config.BATCH_SIZE}")
    print(f"Pixel Size: {config.SQUARE_SIZE} µm")
    print("=" * 80 + "\n")
    
    # Load common m/z data and name mapping
    pivot_df, mz_name_map = load_common_mz_data(config.COMMON_MZS_CSV, config.MSI_ORDER)
    
    # Prepare all tasks
    tasks = []
    for sample_id in config.SAMPLE_IDS:
        if sample_id not in pivot_df.index:
            print(f"⚠️ Sample '{sample_id}' not in pivot table — skipping.")
            continue
        
        # Iterate through columns to get both m/z values and names
        for common_group_name in pivot_df.columns:
            mz_val = pivot_df.loc[sample_id, common_group_name]
            
            # Skip NaN values
            if pd.isna(mz_val):
                continue
            
            mz_val = float(mz_val)
            tasks.append((sample_id, mz_val, common_group_name, config))
    
    print(f"Total tasks to process: {len(tasks)}\n")
    print("=" * 80)
    print("PROCESSING")
    print("=" * 80 + "\n")
    
    # Process in batches to manage memory
    success_count = 0
    fail_count = 0
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = [executor.submit(process_mz_wrapper, task) for task in tasks]
        
        # Process results
        for i, future in enumerate(as_completed(futures), 1):
            success, msg = future.result()
            if success:
                success_count += 1
            else:
                fail_count += 1
            
            print(f"[{i}/{len(tasks)}] {msg}")
            
            # Periodic garbage collection
            if i % config.BATCH_SIZE == 0:
                force_garbage_collection()
                mem_gb = check_memory_usage()
                print(f"  └─ Memory usage: {mem_gb:.2f} GB")
    
    # Final cleanup
    force_garbage_collection()
    
    print("\n" + "=" * 80)
    print("PROCESSING COMPLETE")
    print("=" * 80)
    print(f"✅ Success: {success_count}")
    print(f"⚠️ Failed: {fail_count}")
    print(f"📊 Total: {len(tasks)}")
    print("=" * 80)

if __name__ == "__main__":
    main()

MSI PROCESSING PIPELINE - EXACT DPI CALCULATION
Max Memory: 160 GB
CPU Cores: 78
Batch Size: 156
Pixel Size: 60 µm

Loading common m/z data...
✓ Loaded m/z data for 16 samples

Total tasks to process: 8448

PROCESSING

✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1





✓ Loaded yc_1


✓ Loaded yc_1
✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1
✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1
✓ Loaded yc_1✓ Loaded yc_1
✓ Loaded yc_1
✓ Loaded yc_1✓ Loaded yc_1

✓ Loaded yc_1✓ Loaded yc_1


✓ Loaded yc_1✓ Loaded yc_1










✓ Loaded yc_1
✓ Loaded yc_1














✓ Loaded yc_1
✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1✓ Loaded yc_1






✓ Loaded yc_1✓ Loaded yc_